# 🗂️ Blackboard Research Team: Dynamic Routing with Loop-Backs

A four-agent research and report-drafting team (**Researcher**, **Fact-Checker**, **Outline-Writer**,
**Final-Writer**) collaborates on an open-ended question through a shared blackboard. The path through the agents
depends on what the blackboard contains at each step, including **loop-backs** when the Fact-Checker flags a
finding.

> **No live research here**: nothing is searched. The Researcher writes from the model's own knowledge, and the
> Fact-Checker is the same model judging each finding for clarity, specificity and internal consistency. It does not
> check claims against sources, so a fluent but wrong finding can still come out `verified`. To ground the team,
> give the Researcher a search tool (a `create_agent` loop with `TavilySearch`, as in `01_blackboard_intro.ipynb`)
> and have the Fact-Checker compare each claim with the sources the Researcher retrieved.

## Learning Objectives
In this notebook, you will learn:
1. **Structured contributions** - Post typed entries (`agent`, `type`, `content`) instead of free text
2. **Specialist roles** - Four narrow agents that never decide what happens next
3. **Controller routing** - An LLM controller with a `Literal` decision schema and explicit completion criteria
4. **Trust, but verify** - Code-checked preconditions that overrule an impossible choice, plus a turn cap
5. **Reading the path** - Count loop-backs and overrides from the final blackboard

## Where this fits
`01_blackboard_intro.ipynb` contrasted a blackboard with a fixed pipeline. This notebook uses the same pattern
on a longer task, where the order of work, and any redo, cannot be known in advance. `03_debate.ipynb` then
swaps cooperation for structured disagreement.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | HEARSAY-II speech-recognition system (Erman et al., 1980) — the canonical Blackboard architecture |

### Definition

A **Blackboard System** is a multi-agent architecture in which several specialist agents
collaborate by reading from, and writing to, a shared, central data structure — the
*blackboard*. Instead of a fixed pipeline of hand-offs, a **Controller** inspects the current
state of the blackboard after every contribution and decides, from scratch each time, which
specialist should act next (or whether the task is complete).

### High-level Workflow

1. **Shared Memory (the Blackboard):** A running list of structured contributions — the research
   question, findings, fact-checks, outlines, and drafts — visible to every agent.
2. **Specialist Agents:** Independent agents, each with one narrow responsibility, that read the
   blackboard for context and post a new, structured contribution when activated.
3. **Controller:** A central agent that reads the whole blackboard, reasons about what is missing
   or unresolved, and emits a structured decision: which specialist to call next, or `FINISH`.
4. **Opportunistic Activation:** Only the specialist chosen by the Controller runs. Its output is
   appended to the blackboard, and control returns to the Controller.
5. **Iteration:** This repeats — including looping *back* to an earlier specialist when new
   information demands it — until the Controller judges the task complete.

### When to Use This Pattern

- The right order of operations **cannot be known in advance** — it depends on intermediate
  results (e.g., a fact-check failing and forcing a redo of research).
- Multiple specialists need to **converge on one shared artifact** rather than pass a baton once.
- You want **emergent, opportunistic problem-solving** instead of a rigid DAG of fixed hand-offs.

### Strengths & Weaknesses

**Strengths**
- Naturally handles conditional / non-linear task logic that a fixed sequential graph handles poorly.
- New specialists can be added without redesigning the whole flow — the Controller just needs to
  know they exist.
- The blackboard doubles as a transparent audit trail of *how* the answer was produced.

**Weaknesses**
- Correctness hinges entirely on the Controller's judgment — a weak or under-specified Controller
  prompt can loop forever or finish prematurely.
- Harder to reason about and test than a fixed pipeline, since the execution path is not static.
- Every controller turn is an extra LLM call, adding latency and cost versus a hardcoded router.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY`, imports the building blocks and sets the loop guards.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from typing import List, Literal, Optional, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Loop guards: the controller can legitimately cycle (Researcher <-> Fact-Checker)
MAX_TURNS = 12          # specialist contributions before the controller is forced to FINISH
RECURSION_LIMIT = 40    # LangGraph's hard cap on graph steps (backstop)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the controller and all four specialists
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗂️ Part 1: The Shared Blackboard

**What we are going to do:**
Define the `Contribution` shape every agent writes in, and the `BlackboardState` that threads
through the LangGraph graph. Contributions are structured dicts (`agent`, `type`, `content`)
rather than free-form strings, so the Controller (and we, reading the trace) can reason about
*what kind* of work has been done, not just that "something" was posted.

In [ ]:
# ============================================================================
# THE SHARED BLACKBOARD: Contribution shape, state and helpers
# ============================================================================
class Contribution(TypedDict):
    """One structured entry posted to the shared blackboard."""

    agent: str
    type: Literal["finding", "fact_check", "outline", "draft"]
    content: str


class BlackboardState(TypedDict):
    research_question: str
    blackboard: List[Contribution]
    available_agents: List[str]
    next_agent: Optional[str]
    controller_reasoning: Optional[str]
    iterations: int
    overrides: int                     # times code overruled the controller's choice


def format_blackboard(blackboard: List[Contribution]) -> str:
    """Render the blackboard as readable context for any agent's prompt."""
    if not blackboard:
        return "(empty -- no contributions yet)"
    lines = []
    for i, c in enumerate(blackboard, start=1):
        lines.append(f"{i}. [{c['agent']} | {c['type']}] {c['content']}")
    return "\n".join(lines)


def post(state: BlackboardState, agent: str, kind: str, content: str) -> dict:
    """Helper for a specialist node to append one contribution and bump the turn counter."""
    contribution: Contribution = {"agent": agent, "type": kind, "content": content}
    return {
        "blackboard": state["blackboard"] + [contribution],
        "iterations": state["iterations"] + 1,
    }

print("✅ Blackboard ready. An empty board renders as:", format_blackboard([]))

---
## 👥 Part 2: Specialist Agents

**What we are going to do:**
Build four narrow specialists that each read the blackboard for context and post exactly one
new contribution when activated:

- **Researcher** — contributes a new finding, or revises one the Fact-Checker flagged.
- **Fact-Checker** — reviews the most recent unchecked finding and returns a structured verdict.
- **Outline-Writer** — once enough verified findings exist, organizes them into a report outline.
- **Final-Writer** — drafts the final report following the outline and the verified findings.

None of these agents decides what happens *next* — that is the Controller's job alone. None of them has a
search tool either: the Researcher and the Fact-Checker both work from the model's knowledge (see the note at
the top).

In [ ]:
# ============================================================================
# SPECIALIST AGENTS: Researcher, Fact-Checker, Outline-Writer, Final-Writer
# ============================================================================
def researcher_node(state: BlackboardState) -> dict:
    print("--- RESEARCHER: investigating ---")
    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=(
            "You are the Researcher on a small research team investigating the user's question. "
            "Read the shared blackboard below.\n\n"
            "If the most recent 'fact_check' entry has verdict 'needs_revision', address that "
            "SPECIFIC concern with a corrected or better-sourced finding -- do not repeat the "
            "flawed claim.\n"
            "Otherwise, contribute a NEW finding that has not already been covered on the "
            "blackboard.\n\n"
            "Write 3-5 sentences. Be specific and note any real uncertainty honestly -- do not "
            "invent precise statistics you are not confident about.\n\n"
            f"Blackboard so far:\n{context}"
        )
    )
    human = HumanMessage(content=f"Research question: {state['research_question']}")
    response = llm.invoke([system, human])
    return post(state, "Researcher", "finding", response.content)


class FactCheckResult(BaseModel):
    """Structured verdict on the most recent research finding."""

    target_summary: str = Field(description="Short paraphrase of the finding being checked")
    verdict: Literal["verified", "needs_revision"] = Field(
        description="'verified' if the finding is well-supported and clearly stated, "
        "'needs_revision' if it overstates certainty, is vague, or looks unsupported"
    )
    notes: str = Field(description="Why it was verified, or precisely what must be fixed")


fact_checker_llm = llm.with_structured_output(FactCheckResult)


def fact_checker_node(state: BlackboardState) -> dict:
    print("--- FACT-CHECKER: reviewing latest finding ---")
    findings = [c for c in state["blackboard"] if c["type"] == "finding"]
    latest_finding = findings[-1]
    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=(
            "You are the Fact-Checker on a small research team. Critically evaluate the MOST "
            "RECENT finding below for clarity, specificity, and whether its claims are "
            "reasonably supported. Be skeptical of vague hedging or overconfident claims.\n\n"
            f"Full blackboard for context:\n{context}"
        )
    )
    human = HumanMessage(
        content=f"Finding to review:\n[{latest_finding['agent']}] {latest_finding['content']}"
    )
    result = fact_checker_llm.invoke([system, human])
    content = f"Target: {result.target_summary}\nVerdict: {result.verdict}\nNotes: {result.notes}"
    return post(state, "Fact-Checker", "fact_check", content)


def outline_writer_node(state: BlackboardState) -> dict:
    print("--- OUTLINE-WRITER: structuring the report ---")
    verified_findings = _verified_findings(state["blackboard"])
    context = "\n\n".join(f"- {f['content']}" for f in verified_findings)
    system = SystemMessage(
        content=(
            "You are the Outline-Writer. Using only the VERIFIED findings below, produce a "
            "short numbered outline (3-5 sections) for a report answering the research "
            "question. Each section should name the finding(s) it will cover."
        )
    )
    human = HumanMessage(
        content=f"Research question: {state['research_question']}\n\nVerified findings:\n{context}"
    )
    response = llm.invoke([system, human])
    return post(state, "Outline-Writer", "outline", response.content)


def final_writer_node(state: BlackboardState) -> dict:
    print("--- FINAL-WRITER: drafting the report ---")
    verified_findings = _verified_findings(state["blackboard"])
    outline = [c for c in state["blackboard"] if c["type"] == "outline"][-1]
    findings_text = "\n\n".join(f"- {f['content']}" for f in verified_findings)
    system = SystemMessage(
        content=(
            "You are the Final-Writer. Write a concise final report (roughly 150-250 words) "
            "that follows the outline below and incorporates the verified findings. Use short "
            "headings that mirror the outline sections."
        )
    )
    human = HumanMessage(
        content=(
            f"Research question: {state['research_question']}\n\n"
            f"Outline:\n{outline['content']}\n\n"
            f"Verified findings:\n{findings_text}"
        )
    )
    response = llm.invoke([system, human])
    return post(state, "Final-Writer", "draft", response.content)


def _verified_findings(blackboard: List[Contribution]) -> List[Contribution]:
    """Findings whose own fact-check says 'verified'.

    A finding's own fact-check is the first 'fact_check' posted after it and before the next finding. A revised
    finding is a NEW entry with its own check; the flawed original stays unverified.
    """
    verified = []
    for i, entry in enumerate(blackboard):
        if entry["type"] != "finding":
            continue
        for later in blackboard[i + 1:]:
            if later["type"] == "finding":
                break
            if later["type"] == "fact_check":
                if "Verdict: verified" in later["content"]:
                    verified.append(entry)
                break
    return verified

print("✅ Four specialists defined: Researcher, Fact-Checker, Outline-Writer, Final-Writer")

---
## 🧠 Part 3: The Controller - Dynamic Routing

**What we are going to do:**
Build the Controller as an LLM call with a structured output schema (`ControllerDecision`). It receives the
*entire* blackboard plus the completion criteria and decides which specialist acts next, or whether to `FINISH`.
No order like `Researcher -> Fact-Checker -> Outline-Writer -> Final-Writer` is hard-coded.

An LLM can still make an impossible choice, such as calling the Fact-Checker on an empty board or finishing
without a draft. `precondition_problem()` checks each choice in code, and `rule_based_next()` (the routing rules
written as plain Python) supplies the replacement. Every override is printed and counted in `overrides`.

In [ ]:
# ============================================================================
# THE CONTROLLER: LLM decision + code-checked preconditions
# ============================================================================
class ControllerDecision(BaseModel):
    """The Controller's routing decision for the next graph step."""

    next_agent: Literal[
        "Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer", "FINISH"
    ] = Field(description="Which specialist should act next, or FINISH if the task is complete")
    reasoning: str = Field(description="A brief explanation of why, grounded in the blackboard")


controller_llm = llm.with_structured_output(ControllerDecision)

CONTROLLER_SYSTEM_PROMPT = """You are the Controller of a Blackboard multi-agent research system.
Your team has four specialists, available to you as: {agents}.

Roles:
- Researcher: posts a 'finding' contribution. Can also revise a finding the Fact-Checker flagged.
- Fact-Checker: reviews the most recent unchecked finding and posts a 'fact_check' contribution
  with verdict 'verified' or 'needs_revision'.
- Outline-Writer: once there are at least 2 VERIFIED findings, posts an 'outline' contribution.
- Final-Writer: once an outline exists, posts the final 'draft' contribution.

Completion criteria -- only choose FINISH when ALL of the following hold:
1. There are at least 2 'finding' contributions whose most recent matching 'fact_check' has
   verdict 'verified' (no unresolved 'needs_revision').
2. An 'outline' contribution exists, based on those verified findings.
3. A 'draft' contribution exists that follows the outline.

Routing rules of thumb (apply judgment, do not follow blindly if the blackboard suggests
otherwise):
- If the latest finding has not yet been fact-checked, call the Fact-Checker.
- If the latest fact_check says 'needs_revision', call the Researcher to fix that specific issue
  before doing anything else.
- If fewer than 2 verified findings exist, call the Researcher for another finding.
- Once 2+ verified findings exist and there is no outline yet, call the Outline-Writer.
- Once an outline exists and there is no draft yet, call the Final-Writer.
- Once a draft exists that satisfies the outline, choose FINISH.

Always ground your decision in the actual contents of the blackboard below, not in what "should"
happen next by default."""


def rule_based_next(blackboard: List[Contribution]) -> str:
    """The routing rules of thumb, in plain code. Used when the LLM's choice breaks a precondition."""
    last = blackboard[-1] if blackboard else None
    if last and last["type"] == "finding":
        return "Fact-Checker"
    if last and last["type"] == "fact_check" and "Verdict: needs_revision" in last["content"]:
        return "Researcher"
    if len(_verified_findings(blackboard)) < 2:
        return "Researcher"
    if not any(c["type"] == "outline" for c in blackboard):
        return "Outline-Writer"
    if not any(c["type"] == "draft" for c in blackboard):
        return "Final-Writer"
    return "FINISH"


def precondition_problem(choice: str, blackboard: List[Contribution]) -> Optional[str]:
    """Hard preconditions checked in code. None means the LLM's choice is allowed."""
    if choice == "Fact-Checker" and not any(c["type"] == "finding" for c in blackboard):
        return "there is no finding to check"
    if choice == "Outline-Writer" and len(_verified_findings(blackboard)) < 2:
        return "fewer than 2 verified findings"
    if choice == "Final-Writer" and not any(c["type"] == "outline" for c in blackboard):
        return "there is no outline yet"
    if choice == "FINISH" and not any(c["type"] == "draft" for c in blackboard):
        return "there is no draft yet"
    return None


def controller_node(state: BlackboardState) -> dict:
    if state["iterations"] >= MAX_TURNS:
        print(f"--- CONTROLLER: {MAX_TURNS} contributions reached -> FINISH ---")
        return {"next_agent": "FINISH", "controller_reasoning": "turn cap reached"}

    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=CONTROLLER_SYSTEM_PROMPT.format(agents=state["available_agents"])
    )
    human = HumanMessage(
        content=(
            f"Research question: {state['research_question']}\n\n"
            f"Blackboard (turn {state['iterations']}):\n{context}\n\n"
            "Which agent should act next?"
        )
    )
    decision = controller_llm.invoke([system, human])
    problem = precondition_problem(decision.next_agent, state["blackboard"])
    if problem:
        fallback = rule_based_next(state["blackboard"])
        print(f"--- CONTROLLER: LLM chose {decision.next_agent}, but {problem} -> {fallback} (code override) ---")
        return {
            "next_agent": fallback,
            "controller_reasoning": f"override: {problem}",
            "overrides": state["overrides"] + 1,
        }
    print(f"--- CONTROLLER: -> {decision.next_agent} | {decision.reasoning} ---")
    return {"next_agent": decision.next_agent, "controller_reasoning": decision.reasoning}

print("✅ Controller defined: LLM decision + code-checked preconditions + rule-based fallback")

### 3.1 🧪 Check the Guards Without an LLM

An override only happens when the model makes an impossible choice, and a real run may never show one. The cell
below feeds hand-written blackboards straight to the pure functions, so the guards and the fallback rules are checked
with no API call.

In [ ]:
# ============================================================================
# THE GUARDS ON HAND-WRITTEN BLACKBOARDS (no LLM call)
# ============================================================================
f1 = {"agent": "Researcher", "type": "finding", "content": "Finding 1"}
f1_check = {"agent": "Fact-Checker", "type": "fact_check", "content": "Target: 1\nVerdict: needs_revision\nNotes: vague"}
f2 = {"agent": "Researcher", "type": "finding", "content": "Finding 1, revised"}
f2_check = {"agent": "Fact-Checker", "type": "fact_check", "content": "Target: 1\nVerdict: verified\nNotes: clear"}
demo_board = [f1, f1_check, f2, f2_check]

guard_checks = {
    "Fact-Checker on an empty board is refused": precondition_problem("Fact-Checker", []) == "there is no finding to check",
    "Outline-Writer with 1 verified finding is refused": precondition_problem("Outline-Writer", demo_board) == "fewer than 2 verified findings",
    "FINISH without a draft is refused": precondition_problem("FINISH", demo_board) == "there is no draft yet",
    "only the revised finding counts as verified": _verified_findings(demo_board) == [f2],
    "fallback after an unchecked finding -> Fact-Checker": rule_based_next([f1]) == "Fact-Checker",
    "fallback after a needs_revision check -> Researcher": rule_based_next([f1, f1_check]) == "Researcher",
}
for name, ok in guard_checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

---
## 🏗️ Part 4: Assembling the Graph

**What we are going to do:**
Wire the Controller as the entry point and hub of the graph. Every specialist edges back to the
Controller after posting its contribution; the Controller's structured decision determines the
next hop via a conditional edge -- including `FINISH`, which routes to `END`.

In [ ]:
# ============================================================================
# ASSEMBLE THE GRAPH
# ============================================================================
def route_from_controller(state: BlackboardState) -> str:
    return state["next_agent"]


graph_builder = StateGraph(BlackboardState)

graph_builder.add_node("Controller", controller_node)
graph_builder.add_node("Researcher", researcher_node)
graph_builder.add_node("Fact-Checker", fact_checker_node)
graph_builder.add_node("Outline-Writer", outline_writer_node)
graph_builder.add_node("Final-Writer", final_writer_node)

graph_builder.add_edge(START, "Controller")

graph_builder.add_conditional_edges(
    "Controller",
    route_from_controller,
    {
        "Researcher": "Researcher",
        "Fact-Checker": "Fact-Checker",
        "Outline-Writer": "Outline-Writer",
        "Final-Writer": "Final-Writer",
        "FINISH": END,
    },
)

for specialist in ["Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer"]:
    graph_builder.add_edge(specialist, "Controller")

blackboard_app = graph_builder.compile()
print("✅ Blackboard graph compiled. Edges:")
for edge in blackboard_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

---
## ▶️ Part 5: Running the Blackboard on a Non-Trivial Research Question

**What we are going to do:**
Pose an open-ended research question with no single "obvious" answer, then stream the graph
turn-by-turn, printing the Controller's decision and each specialist's contribution as they are
posted. Watch for the Controller looping back to the Researcher if the Fact-Checker flags a
finding -- that loop-back is not scripted anywhere in this notebook; it emerges from the
Controller re-reading the blackboard each turn.

In [ ]:
# ============================================================================
# RUN THE BLACKBOARD, PRINTING EACH NEW CONTRIBUTION
# ============================================================================
research_question = (
    "What are the most effective strategies for keeping LLM-based agents reliable over "
    "long-horizon, multi-step tasks, and what open problems remain?"
)

initial_state: BlackboardState = {
    "research_question": research_question,
    "blackboard": [],
    "available_agents": ["Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer"],
    "next_agent": None,
    "controller_reasoning": None,
    "iterations": 0,
    "overrides": 0,
}

final_state, seen = None, 0
for state in blackboard_app.stream(initial_state, {"recursion_limit": RECURSION_LIMIT}, stream_mode="values"):
    final_state = state
    for entry in state["blackboard"][seen:]:
        print(f"\n[{entry['agent']} | {entry['type']}]\n{entry['content']}\n")
    seen = len(state["blackboard"])

In [ ]:
# ============================================================================
# THE FINAL REPORT AND A CODE-COMPUTED SUMMARY OF THE PATH
# ============================================================================
board = final_state["blackboard"]
drafts = [c for c in board if c["type"] == "draft"]
print(drafts[-1]["content"] if drafts else "No draft was posted (the turn cap stopped the run first).")

loop_backs = sum(
    1 for prev, cur in zip(board, board[1:])
    if prev["type"] == "fact_check" and "Verdict: needs_revision" in prev["content"] and cur["type"] == "finding"
)
print("\n--- Path summary ---")
print("Sequence:           ", " -> ".join(c["agent"] for c in board))
print("Contributions:      ", len(board))
print("Verified findings:  ", len(_verified_findings(board)))
print("Loop-backs:         ", loop_backs, "(a needs_revision check followed by a new finding)")
print("Controller overrides:", final_state["overrides"])

**Reading the output:** the `Sequence` line is the path the controller chose. A run with `Loop-backs: 0` means
every finding passed its first fact-check. That is a valid run, not a failure: the loop-back only happens when
the Fact-Checker actually flags something, which is the point of dynamic routing. `Controller overrides` counts
the times the LLM named a specialist whose precondition was not met, and code substituted the rule-based choice.
The exact path varies between runs because every step is an LLM judgment.

---
## 📝 Summary

- A **Blackboard system** replaces fixed hand-offs with a shared, structured store that every specialist reads
  and writes.
- The **Controller** alone decides sequencing, by re-reading the blackboard every turn.
- **Structured contributions** (`agent`, `type`, `content`) let code, not just the LLM, reason about what has been
  done. `_verified_findings()` pairs each finding with its own fact-check.
- **Code-checked preconditions** and a **turn cap** keep an LLM controller from calling impossible steps or
  looping forever. The rule-based fallback shows the same routing logic written as plain code.
- Dynamism shows up as **loop-backs**: a `needs_revision` verdict sends control back to the Researcher.
- **Grounding is the missing piece**: without a search tool, the "fact-check" tests clarity and consistency,
  not truth. Give the Researcher sources before trusting a `verified` label.

### Next Steps
- `03_debate.ipynb`: agents that *argue* opposing positions, with a judge that synthesizes the result